<a href="https://colab.research.google.com/github/BlackCatSpb/widebind-mini/blob/master/notebooks/colab_mini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WideBind Mini — Colab Training (T4, 16 GB)
12.23M params, 12 layers, D=896, G=8 experts.
Private Memory + Staircase k + 4-scale VSA + BottleneckBind shift.

**Before running:** upload to Google Drive:
- `mini_code/` — папка с `core/`, `scripts/`, `train.py`
- `token_stream_*_clean.bin` — данные

Или просто всё в корне `widebind_data/` без подпапки.

In [1]:
# --- 1. Mount Drive ---
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/widebind_data'

Mounted at /content/drive


In [2]:
# --- 2. Find code folder ---
import sys, os, shutil, glob, gc, time, math, re
import torch
import torch.nn.functional as F
import numpy as np

SRC = None
for d in [os.path.join(DRIVE, 'mini_code'), DRIVE]:
    if os.path.isdir(os.path.join(d, 'core')) and os.path.isfile(os.path.join(d, 'train.py')):
        SRC = d; break
if SRC is None:
    print('Searching widebind_data/ for code...')
    for root, dirs, _ in os.walk(DRIVE, topdown=True):
        for d in dirs:
            p = os.path.join(root, d)
            if os.path.isdir(os.path.join(p, 'core')) and os.path.isfile(os.path.join(p, 'train.py')):
                SRC = p; break
        if SRC: break
if SRC is None:
    print(f'Contents of {DRIVE}:')
    for item in os.listdir(DRIVE):
        print(f'  {item}/' if os.path.isdir(os.path.join(DRIVE, item)) else f'  {item}')
    raise FileNotFoundError(f'mini_code/ or core/ not found in {DRIVE}')

sys.path.insert(0, SRC)
print(f'Code: {SRC}')
for sd in ['core', 'scripts']:
    print(f'  {sd}/: {"OK" if os.path.isdir(os.path.join(SRC, sd)) else "MISSING"}')
print(f'train.py: {"OK" if os.path.isfile(os.path.join(SRC, "train.py")) else "MISSING"}')

Searching widebind_data/ for code...
Code: /content/drive/MyDrive/widebind_data/WideBand Mini
  core/: OK
  scripts/: OK
train.py: OK


In [3]:
# --- 3. Imports ---
from core import WideBandConfig, WideBindStack, MirrorLRScheduler, AdaptiveController

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cuda


In [4]:
# --- 4. Config ---
cfg = WideBandConfig(
    D=896, n_layers=12, bind_K=32,
    mlp_groups=8, mlp_expand=4,
    vocab=50000, seq_len=512, lr=3e-4, max_steps=300000,
    grad_clip=0.5, conv_kernel=48,
    data_dir=DRIVE,
    save_dir=os.path.join(DRIVE, 'checkpoints'),
    log_dir=os.path.join(DRIVE, 'logs'),
    bind_twist_mode='shift',
    accum_steps=8,
    private_mem=True,
    div_weight=0.0001,
    ranking_weight=0.01,
    signal_entropy_weight=0.001,
    log_scale_l2_weight=0.01,
)
# lambda_d_enabled=True (default) — переопределяет warmup/eval/save интервалы
print(f'D={cfg.D} L={cfg.n_layers} G={cfg.mlp_groups} K={cfg.bind_K} seq={cfg.seq_len}')
print(f'  warmup={cfg.warmup_steps} eval={cfg.eval_interval} save={cfg.save_interval}')

D=896 L=12 G=8 K=32 seq=512
  warmup=101 eval=233 save=987


In [5]:
# --- 5. Model on CPU (sanity) ---
model = WideBindStack(cfg)
n = model.param_count()
print(f'Params: {n:,} ({n/1e6:.2f}M)')

x = torch.randint(0, cfg.vocab, (1, 16))
h = model.embed_tokens(x)
out, state, gs = model(h)
assert out.shape[-1] == cfg.D
print(f'Forward: OK, out.std={out.std():.4f}')
del x, h, out, state, gs; gc.collect()

Params: 12,348,196 (12.35M)
Forward: OK, out.std=0.9998


30

In [6]:
# --- 6. Data streams ---
stream_files = sorted(glob.glob(os.path.join(DRIVE, 'token_stream_*_clean.bin')))
if not stream_files:
    stream_files = sorted(glob.glob(os.path.join(DRIVE, 'token_stream_*.bin')))
if not stream_files:
    raise FileNotFoundError(f'token_stream_*.bin not found in {DRIVE}')
streams = [np.memmap(f, dtype=np.uint16, mode='r') for f in stream_files]
total_tok = sum(len(s) for s in streams)
print(f'Streams: {len(streams)} files, {total_tok:,} tokens')

class StreamReader:
    def __init__(self, streams, S, B):
        self.streams = streams; self.S = S; self.B = B
        self.si = 0; self.off = 0
    def get_batch(self):
        s = self.streams[self.si]
        need = self.B * self.S + 1
        if self.off + need > len(s):
            self.off = 0; self.si = (self.si + 1) % len(self.streams)
            s = self.streams[self.si]
        ch = s[self.off:self.off+need]
        x = torch.from_numpy(ch[:self.B*self.S].copy()).long().view(self.B, self.S)
        y = torch.from_numpy(ch[1:self.B*self.S+1].copy()).long().view(self.B, self.S)
        self.off += self.B * self.S
        return x, y

Streams: 2 files, 3,964,570,754 tokens


In [7]:
# --- 7. Batch size ---
B = 2
cfg.batch_size = B
reader = StreamReader(streams, cfg.seq_len, B)
print(f'Batch size: {B}')

Batch size: 2


In [8]:
# --- 8. Model to GPU + Optimizer (FP32) ---
model = model.to(device)
groups = model.param_groups()
optimizer = torch.optim.AdamW(groups, betas=(0.9, 0.95))
scheduler = MirrorLRScheduler(model, optimizer, cfg=cfg)
accum = getattr(cfg, 'accum_steps', 1)
mem_free, mem_total = torch.cuda.mem_get_info()
print(f'GPU: {mem_free/1e9:.2f}/{mem_total/1e9:.2f} GB free')
print(f'Optimizer ready, accum={accum}')

GPU: 15.41/15.64 GB free
Optimizer ready, accum=8


In [9]:
# --- 9. Resume ---
start_step = 0; best_val = float('inf')
os.makedirs(cfg.save_dir, exist_ok=True)

def step_key(p):
    m = re.search(r'step_(\d+)', os.path.basename(p))
    return int(m.group(1)) if m else 0

ckpts = sorted(glob.glob(os.path.join(cfg.save_dir, '*.pt')), key=step_key)
loaded = False
if ckpts:
    try:
        ckpt = torch.load(ckpts[-1], map_location='cpu', weights_only=False)
        assert 'model' in ckpt, f'Keys: {list(ckpt.keys())[:5]}'
        miss, unexp = model.load_state_dict(ckpt['model'], strict=False)
        if len(miss) > 10:
            print(f'  MISS({len(miss)}): {miss[:5]}...')
        if unexp:
            print(f'  UNEXP({len(unexp)}): {unexp[:3]}...')
        if 'optimizer' in ckpt:
            try: optimizer.load_state_dict(ckpt['optimizer'])
            except Exception as e: print(f'  Opt load: {e}')
        if 'scheduler' in ckpt:
            try: scheduler.load_state_dict(ckpt['scheduler'])
            except Exception as e: print(f'  Sched load: {e}')
        start_step = ckpt.get('step', 0)
        best_val = ckpt.get('best_val_loss', float('inf'))
        print(f'Resumed: step {start_step}, best={best_val:.4f}, miss={len(miss)}')
        with torch.no_grad():
            xt = torch.randint(0, cfg.vocab, (B, cfg.seq_len), device=device)
            ht = model.embed_tokens(xt)
            ot, _, _ = model(ht)
            ls = model.compute_loss(ot, xt[:, 1:])
            if torch.isnan(ot).any() or torch.isnan(ls):
                raise RuntimeError(f'NaN after load!')
            print(f'  Sanity: out.std={ot.std():.4f} loss={ls.item():.4f}')
        del ckpt, xt, ht, ot, ls; gc.collect()
        loaded = True
    except Exception as e:
        print(f'Corrupted, fresh start: {e}')
if not loaded:
    print('Fresh start')

Resumed: step 425, best=14.9706, miss=0
Corrupted, fresh start: Expected input batch_size (1024) to match target batch_size (1022).
Fresh start


In [10]:
# --- 10. Eval ---
@torch.no_grad()
def evaluate():
    model.eval()
    total, cnt = 0.0, 0
    n = max(1, sum(len(s) for s in streams) // (B * cfg.seq_len) // len(streams))
    n = min(100, n)
    for si, s in enumerate(streams):
        off = max(len(s) // 4, B * cfg.seq_len + 1)
        state = None; gs = None
        for _ in range(n):
            if off + B * cfg.seq_len + 1 > len(s): off = 0; break
            ch = s[off:off+B*cfg.seq_len+1]
            x = torch.from_numpy(ch[:B*cfg.seq_len].copy()).long().view(B, cfg.seq_len).to(device)
            y = torch.from_numpy(ch[1:B*cfg.seq_len+1].copy()).long().view(B, cfg.seq_len).to(device)
            off += B * cfg.seq_len
            h = model.embed_tokens(x)
            out, state, gs = model(h, state, global_state=gs, adaptive=False)
            total += model.compute_loss(out, y).item(); cnt += 1
            if _ % 25 == 24: state = None; gs = None
        del state, gs; gc.collect()
    model.train()
    return total / max(cnt, 1)

In [11]:
# --- 11. NaN Diagnostic ---
x_diag, y_diag = reader.get_batch()
x_diag = x_diag.to(device); y_diag = y_diag.to(device)
t_min = x_diag.min().item(); t_max = x_diag.max().item()
print(f'  Token range: [{t_min}, {t_max}] (vocab={cfg.vocab})')
if t_max >= cfg.vocab:
    raise ValueError(f'Token {t_max} >= vocab {cfg.vocab}! Data has OOB tokens.')
if t_min < 0:
    raise ValueError(f'Token {t_min} < 0! Data is corrupted.')
with torch.no_grad():
    h_diag = model.embed_tokens(x_diag)
    print(f'  embed: out.std={h_diag.std():.4f} nan={torch.isnan(h_diag).any().item()}')
    s_diag = [None] * len(model.layers)
    gs_diag = torch.zeros(len(model.layers), 1, cfg.D, device=device)
    for i, (layer, s) in enumerate(zip(model.layers, s_diag)):
        h_diag, s_out = layer(h_diag, s, global_state=gs_diag[i:i+1].clone())
        nan = torch.isnan(h_diag).any().item()
        inf = torch.isinf(h_diag).any().item()
        std = h_diag.std().item()
        print(f'  layer {i:>2}: std={std:.4f} nan={nan} inf={inf}')
        if nan or inf:
            raise RuntimeError(f'NaN/Inf at layer {i}')
        s_diag[i] = s_out
    out_diag = F.rms_norm(h_diag, (cfg.D,), model.final_norm_w)
    loss_diag = model.compute_loss(out_diag, y_diag)
    print(f'  final: std={out_diag.std():.4f} loss={loss_diag.item():.4f}')
    if torch.isnan(out_diag).any() or torch.isnan(loss_diag):
        raise RuntimeError('NaN in final output or loss')
print('=== Diagnostic PASS ===')
del x_diag, y_diag, h_diag, out_diag, loss_diag, s_diag, gs_diag; gc.collect()
torch.cuda.empty_cache()

  Token range: [10, 43034] (vocab=50000)
  embed: out.std=0.0111 nan=False
  layer  0: std=10.4304 nan=False inf=False
  layer  1: std=5.1816 nan=False inf=False
  layer  2: std=8.0527 nan=False inf=False
  layer  3: std=2.7051 nan=False inf=False
  layer  4: std=20.1269 nan=False inf=False
  layer  5: std=55.0660 nan=False inf=False
  layer  6: std=132.9931 nan=False inf=False
  layer  7: std=2.3763 nan=False inf=False
  layer  8: std=3.4298 nan=False inf=False
  layer  9: std=23.6385 nan=False inf=False
  layer 10: std=1631.1578 nan=False inf=False
  layer 11: std=26.1695 nan=False inf=False
  final: std=0.9994 loss=11.3572
=== Diagnostic PASS ===


In [ ]:
# --- 12. Training Loop (FP32) ---
state = None; gs = None; t0 = time.time(); tokens = 0

def _detach(x):
    if x is None: return None
    if isinstance(x, torch.Tensor): return x.detach()
    if isinstance(x, (tuple, list)): return type(x)(_detach(v) for v in x)
    return x

print(f'Training: {start_step} to {cfg.max_steps}')
print(f'GPU: {torch.cuda.mem_get_info()[0]/1e9:.2f}/{torch.cuda.mem_get_info()[1]/1e9:.2f} GB')
try:
    for step in range(start_step, cfg.max_steps):
        x, y = reader.get_batch()
        x, y = x.to(device), y.to(device)

        if (y[:, -1] == 2).any() and state is not None:
            state = _detach(state)

        h = model.embed_tokens(x)
        out, state, gs = model(h, state, global_state=gs)
        loss = model.compute_loss(out, y) / accum

        state = _detach(state)
        gs = _detach(gs)

        loss.backward()
        tokens += B * cfg.seq_len

        if (step + 1) % accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            scheduler.report_train_loss(loss.item() * accum)

        if step % cfg.log_interval == 0:
            dt = time.time() - t0
            try:
                idiff = torch.stack([(1.0 - l.mirror.alpha_diag.data).abs().mean() for l in model.layers]).mean().item()
                gvar = torch.stack([l.mirror._last_gates.var() for l in model.layers]).mean().item()
                ls_var = torch.stack([l.mirror.log_scale.data.var() for l in model.layers]).mean().item()
            except Exception:
                idiff = gvar = ls_var = 0.0
            lr = scheduler.get_last_lr()[0]
            print(f'step={step:>6} loss={loss.item()*accum:.4f} |1-a|={idiff:.4f} '
                  f'g_var={gvar:.4f} ls_var={ls_var:.4f} lr={lr:.2e} tok/s={tokens/dt:.0f}')

        if step > 0 and step % cfg.eval_interval == 0:
            vl = evaluate()
            print(f'EVAL step={step}: val_loss={vl:.4f} ppl={math.exp(min(vl, 20)):.2f}')
            scheduler.report_val_loss(vl)
            torch.cuda.empty_cache(); gc.collect()
            if vl < best_val:
                best_val = vl
                save_path = os.path.join(cfg.save_dir, 'best.pt')
                try:
                    os.makedirs(cfg.save_dir, exist_ok=True)
                    sd = {k: v.cpu() for k, v in model.state_dict().items()}
                    torch.save({'step': step, 'model': sd,
                               'best_val_loss': best_val, 'cfg': cfg}, save_path)
                    print(f'  New best! Saved to {save_path}')
                except Exception as e:
                    print(f'  SAVE FAILED: {e}')

        if step > 0 and step % cfg.save_interval == 0:
            save_path = os.path.join(cfg.save_dir, f'step_{step}.pt')
            try:
                os.makedirs(cfg.save_dir, exist_ok=True)
                sd = {k: v.cpu() for k, v in model.state_dict().items()}
                ckpt = {'step': step, 'model': sd,
                        'optimizer': optimizer.state_dict(),
                        'scheduler': scheduler.state_dict(),
                        'best_val_loss': best_val, 'cfg': cfg}
                torch.save(ckpt, save_path)
                print(f'Checkpoint saved ({len(ckpt)} keys) -> {save_path}')
                del ckpt, sd; gc.collect()
            except Exception as e:
                print(f'  SAVE FAILED: {e}')

except KeyboardInterrupt:
    save_path = os.path.join(cfg.save_dir, f'interrupt_step_{step}.pt')
    try:
        sd = {k: v.cpu() for k, v in model.state_dict().items()}
        torch.save({'step': step, 'model': sd,
                   'best_val_loss': best_val, 'cfg': cfg}, save_path)
        print(f'\nInterrupted. Saved {save_path}')
    except Exception as e:
        print(f'  SAVE FAILED on interrupt: {e}')

Training: 425 to 300000
GPU: 14.89/15.64 GB
step=   440 loss=11.2362 |1-a|=0.1055 g_var=0.0030 ls_var=0.0376 lr=5.94e-06 tok/s=1301
EVAL step=466: val_loss=11.1601 ppl=70266.76
  New best! Saved to /content/drive/MyDrive/widebind_data/checkpoints/best.pt
step=   495 loss=11.2704 |1-a|=0.1055 g_var=0.0042 ls_var=0.0375 lr=2.67e-05 tok/s=729
step=   550 loss=11.2251 |1-a|=0.1055 g_var=0.0041 ls_var=0.0375 lr=4.46e-05 tok/s=912
step=   605 loss=11.1609 |1-a|=0.1056 g_var=0.0040 ls_var=0.0374 lr=6.53e-05 tok/s=1011
step=   660 loss=11.1105 |1-a|=0.1058 g_var=0.0038 ls_var=0.0373 lr=8.61e-05 tok/s=1072
EVAL step=699: val_loss=11.0510 ppl=63007.73
  New best! Saved to /content/drive/MyDrive/widebind_data/checkpoints/best.pt
step=   715 loss=11.0781 |1-a|=0.1060 g_var=0.0036 ls_var=0.0372 lr=1.07e-04 tok/s=964
step=   770 loss=11.0137 |1-a|=0.1062 g_var=0.0033 ls_var=0.0370 lr=1.28e-04 tok/s=1008
step=   825 loss=10.9380 |1-a|=0.1063 g_var=0.0029 ls_var=0.0368 lr=1.49e-04 tok/s=1040
step=   8